# Silver Layer – Purchase Orders Cleaning

## Ziel

In diesem Schritt werden die Purchase Orders aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Datentypen**, der **Bereinigung von String- und Preisfeldern**, der **Konvertierung des Unix-Timestamps** sowie der **Entfernung nicht benötigter bzw. sensibler Spalten**.

## Durchgeführte Schritte

### 1. Identifikatoren standardisieren

Die EAN-Identifikatoren werden als String gespeichert:

* `EAN13` → `ean13` (`String`)
* `EAN5` → `ean5` (`String`)

Dadurch bleiben beispielsweise führende Nullen in den EAN-Werten erhalten.

### 2. Purchase Order ID standardisieren

Die Spalte `PO` wird in `purchase_order` umbenannt und als Integer gespeichert:

* `PO` → `purchase_order` (`Integer`)

Damit liegt die Purchase-Order-ID in einem einheitlichen numerischen Datentyp vor.

### 3. Promotion-Datum standardisieren

Das Feld `datetime` enthält einen Unix-Timestamp.

Dieser wird in einen Spark Timestamp konvertiert und unter `promotion_datetime` gespeichert.

* `datetime` → `promotion_datetime` (`Timestamp`)

Dadurch kann das Datum später für zeitbasierte Analysen und Berechnungen verwendet werden.

### 4. Preis bereinigen und standardisieren

Das Feld `price` wird zunächst von führenden und nachfolgenden Leerzeichen sowie dem `$`-Zeichen bereinigt.

Anschließend wird der Wert in einen Dezimaldatentyp mit zwei Nachkommastellen konvertiert:

* `price` → `Decimal(18,2)`

Beispiel:

`$ 39` → `39.00`

### 5. String-Felder bereinigen

Die Textfelder werden von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `product_name`
* `product_unit`
* `purchaser`
* `supplier`

Hierfür wird die `trim()`-Funktion verwendet.

### 6. Quantity standardisieren

Das Feld `quantity` wird zunächst bereinigt und anschließend in einen Integer-Datentyp konvertiert:

* `quantity` → `Integer`

Dadurch kann die Menge später für numerische Berechnungen verwendet werden.

### 7. Nicht benötigte und sensible Spalten entfernen

Nach der Transformation werden die ursprünglichen Spalten `password` und `datetime` entfernt.

* `password` wird aus Datenschutz- und Sicherheitsgründen nicht weitergeführt.
* `datetime` wird entfernt, da die Information bereits in `promotion_datetime` übernommen wurde.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(silver_df.limit(5))


In [1]:
from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace,
    from_unixtime   
)
from pyspark.sql import functions as F


bronze_df = spark.table("Retailorg_bronze.purchase_orders")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["ean13", "ean5", "PO"])

silver_df = (df_deduplicated

    # Identifiers -> String
    .withColumn("ean13", F.col("EAN13").cast("string"))\
    .withColumn("ean5", F.col("EAN5").cast("string"))\
  
    # Purchase Order ID

    .withColumn("purchase_order", F.col("PO").cast("int"))\
    # Unix Timestamp -> Timestamp
     .withColumn("promotion_datetime", F.from_unixtime(F.col("datetime")).cast("timestamp"))\

    # Password NICHT übernehmen

    # "$ 39" -> 39.00
    .withColumn("price", regexp_replace(
        trim(col("price")),
        r"[$ ]",
        ""
    ).cast("decimal(18,2)").alias("price"))\

    # Strings bereinigen
    .withColumn("product_name", trim(F.col("product_name")).cast("string"))\
    .withColumn("product_unit", trim(F.col("product_unit")).cast("string"))\
    .withColumn("purchaser", trim(F.col("purchaser")).cast("string"))\

    # Quantity
    .withColumn("quantity", trim(F.col("quantity")).cast("int"))\

    # Supplier
    .withColumn("supplier", trim(F.col("supplier")).cast("string"))

    # drop password and datetime
    .drop("password", "datetime")

)

print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {silver_df.count()}")

display(silver_df.limit(5))

StatementMeta(, 58db54d7-7256-42f5-a847-b2e3f779a497, 3, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 3973 
Datensätze nach Bereinigung (ohne Duplikate): 3973


SynapseWidget(Synapse.DataFrame, f2164808-069c-41ed-bf53-2c6536b9cf3b)


### 1. Die Pflichtspalten (Harte Filter)

Diese Spalten dürfen keine Null-Werte enthalten. Wenn sie leer sind, fliegt die Zeile raus oder geht in die Quarantäne:

 *  **PO** (**oder purchase_order**– je nachdem, welche Spalte Ihre **primäre Bestellnummer** ist).
 * **ean13** (**Ohne den Artikelcode** ist die Zeile unbrauchbar).


### 2. Die Spalten für Finanzmathematik (Ersetzen durch Zahlenwerte)

* quantity: Bei Null ersetzen durch 0. (Verhindert Berechnungsfehler).
* price: Bei Null ersetzen durch 0.00 oder über einen Lookup aus dim_products. (Verhindert, dass der Gesamtumsatz zu Null wird).


### 3. Die Text- und Gruppierungsspalten (Ersetzen durch Standard-Texte)

* **product_name**: Bei Null ersetzen durch "Unbekanntes Produkt".
* **product_unit**: Bei Null ersetzen durch "Stück".
* **purchaser**: Bei Null ersetzen durch "Nicht zugewiesen".
* **supplier**: Bei Null ersetzen durch "Unbekannter Lieferant".

In [11]:
from pyspark.sql.functions import col, coalesce, lit

# Filterung der Pflichtfelder und Behandlung aller Null-Werte
silver_po_final = (silver_df
    # 1. Zeilen ohne Bestellnummer oder Artikelnummer direkt herausfiltern
    .filter(col("PO").isNotNull() & col("ean13").isNotNull())
    
    # 2. Null-Werte durch logische Standards ersetzen
    .select(
        col("PO").alias("purchase_order_id"),
        col("purchase_order"),
        col("ean13"),
        coalesce(col("ean5"), lit("")).alias("ean5"),
        
        # Mengen und Preise vor Null-Werten schützen
        coalesce(col("quantity"), lit(0)).alias("quantity"),
        coalesce(col("price"), lit(0.00)).alias("unit_price"),
        
        # Text-Spalten lesbar machen
        coalesce(col("product_name"), lit("Unbekanntes Produkt")).alias("product_name"),
        coalesce(col("product_unit"), lit("Stück")).alias("order_unit"),
        coalesce(col("purchaser"), lit("System-Einkäufer")).alias("purchaser_name"),
        coalesce(col("supplier"), lit("Unbekannter Lieferant")).alias("supplier_name"),
        
        col("promotion_datetime")
    )
   
)

# Ergebnis in Databricks anzeigen
display(silver_po_final.limit(3))

print(f"Datensätze vor Bereinigung: { silver_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {silver_po_final.count()}")

silver_po_final.printSchema()

StatementMeta(, 58db54d7-7256-42f5-a847-b2e3f779a497, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50b006f4-2076-4301-a58c-71d292ae94f9)

Datensätze vor Bereinigung: 3973 
Datensätze nach Bereinigung (ohne Duplikate): 3973
root
 |-- purchase_order_id: long (nullable = true)
 |-- purchase_order: integer (nullable = true)
 |-- ean13: string (nullable = true)
 |-- ean5: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- product_name: string (nullable = false)
 |-- order_unit: string (nullable = false)
 |-- purchaser_name: string (nullable = false)
 |-- supplier_name: string (nullable = false)
 |-- promotion_datetime: timestamp (nullable = true)



## Das Silver Layer kurz zusammengefasst:

*  **Datenqualität garantiert**: Alle Duplikate, Rechenfehler (durch Null-Werte) und Textfehler wurden dauerhaft beseitigt.

* **Strukturierte Wahrheit**: Die Daten wurden standardisiert, sinnvoll umbenannt und in korrekte Datentypen (wie Decimal und Timestamp) gecastet.

* **Sicher und Bereit**: Sensible Daten (Passwörter) wurden entfernt. Die Tabelle ist nun die perfekte, absturzsichere Startrampe für alle Berichte im Gold Layer.

In [5]:
spark.sql("CREATE SCHEMA IF NOT EXISTS Retailorg_silver")


StatementMeta(, 7c9af660-688a-489a-895d-930399ca1988, 7, Finished, Available, Finished, False)

DataFrame[]

In [12]:
silver_po_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.purchase_orders")


StatementMeta(, 58db54d7-7256-42f5-a847-b2e3f779a497, 14, Finished, Available, Finished, False)

# Silver Layer – Active Promotions Cleaning

## Ziel

In diesem Schritt werden die transaktionsbasierten Werbeaktionen (*Active Promotions*) aus dem Bronze Layer strukturell bereinigt, von Duplikaten befreit und technisch standardisiert.

Der Schwerpunkt liegt auf der **Normalisierung von Aktionszeitstempeln**, dem **Casting numerischer Datentypen** (wie Rabattsätze und Mindestmengen) sowie dem **Ausschluss analytisch redundanter Hilfsspalten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** unter dem fachlich konsistenten Namen `sales_promotion` gespeichert.

---

## Durchgeführte Schritte

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `.dropDuplicates(["promo_id", "promo_datetime", "promo_item"])` wird sichergestellt, dass jede Kombination aus Werbeaktion, Einlösungszeitpunkt und Artikel eindeutig bleibt. Ein fehlerhafter Drop über Lieferanten-IDs wird dadurch vermieden, um echten Datenverlust zu verhindern.

### 2. Identifikatoren standardisieren
Die Identifikatoren werden von überflüssigen Leerzeichen bereinigt, im Fall von fehlenden Werten durch Standard-Platzhalter ersetzt und typisiert:
* `promo_id` → `promotion_id` (`String` - Pflichtfeld, harte Filterung auf `isNotNull`)
* `promo_customer` → `customer_id` (`String` - Verbindung zur Kundentabelle)
* `promo_item` → `product_id` (`String` - Verbindung zum Produktkatalog)

### 3. Zeitstempel und Laufzeiten konvertieren
Die als Freitext/String gelieferten Datums- und Uhrzeitfelder werden in echte, hochperformante Zeitstempel für lückenlose Zeitreihenanalysen umgewandelt:
* `promo_datetime` → `redemption_timestamp` (`Timestamp` - Einlösungszeitpunkt an der Kasse)
* `promo_began` → `campaign_start` (`Timestamp` - Startdatum der Aktion)
* `deadline` → `campaign_end` (`Timestamp` - Letzter Gültigkeitstag der Aktion)

### 4. Rabatte und Mengen absichern (Finanz-Logik)
Mengen, Schwellenwerte und prozentuale Rabattsätze werden in exakte numerische Datentypen konvertiert. Fehlende Werte (`NULL`) werden durch logische Standardwerte ersetzt, um Rechenfehler im Gold Layer zu verhindern:
* `promo_disc` → `percentage_discount_rate` (`Decimal(5,4)` - z. B. `0.1500` für 15% Rabatt. Fallback: `0.0000`)
* `promo_qty` → `discounted_quantity` (`Integer` - Menge der rabattierten Artikel. Fallback: `0`)
* `units_required` → `min_units_required` (`Integer` - Mindestmenge im Warenkorb. Fallback: `1`)

### 5. Status-Flags und Bereinigung von Hilfsspalten
* `eligible` → `is_eligible_flag` (`Integer` - Zeigt an, ob der Kunde berechtigt war: `1` = Ja, `0` = Nein)
* Die Spalte `cumsum` (kumulierte Zwischensumme des Quellsystems) wird durch ein gezieltes Abschluss-`.select()` vollständig verworfen, da sie für analytische Zwecke im Datenmodell redundant ist.

---

## Datenqualität überprüfen

Nach der Transformation werden die Datensatzzahlen vor und nach der Bereinigung im Notebook protokolliert und die ersten fünf Datensätze zur visuellen Kontrolle ausgegeben:

```python
display(silver_df.limit(5))
```


In [7]:
from pyspark.sql.functions import col, trim, to_timestamp, round
from pyspark.sql import functions as F

# 1. Laden der echten CSV-Tabelle aus Bronze
bronze_df = spark.table("Retailorg_bronze.active_promotions")

# 2. Eindeutigkeit sichern: Duplikate über die echten fachlichen Schlüssel entfernen
# Ein Lieferanten-Drop-Duplicate würde hier alle Marketingdaten zerstören!
df_deduplicated = bronze_df.dropDuplicates(["promo_id", "promo_datetime", "promo_item"])

# 3. Transformation & technische Standardisierung
silver_df = (
    df_deduplicated

    # Identifikatoren bereinigen und typisieren
    .withColumn("promotion_id", F.col("promo_id").cast("string"))
    .withColumn("product_id", F.trim(F.col("promo_item")).cast("string"))
    .withColumn("customer_id", F.trim(F.col("promo_customer")).cast("string"))

    # Strings in echte Zeitstempel konvertieren
    .withColumn("campaign_start", F.to_timestamp(F.trim(F.col("promo_began"))))
    .withColumn("campaign_end", F.to_timestamp(F.trim(F.col("deadline"))))
    .withColumn("redemption_timestamp", F.to_timestamp(F.trim(F.col("promo_datetime"))))

    # Numerische Datentypen & Schwellenwerte casten
    .withColumn("is_eligible_flag", F.col("eligible").cast("int"))
    .withColumn("min_units_required", F.col("units_required").cast("int"))
    .withColumn("discounted_quantity", F.col("promo_qty").cast("int"))

    # Prozentualen Rabattsatz hochpräzise casten
    .withColumn("percentage_discount_rate", F.col("promo_disc").cast("decimal(5,4)"))

    # Absolute Bereinigung: Wir behalten NUR noch die neuen, sauberen Spalten
    # Dadurch fliegen 'cumsum' und alle alten 'promo_'-Rohspalten automatisch raus
    .select(
        "promotion_id",
        "product_id",
        "customer_id",
        "campaign_start",
        "campaign_end",
        "redemption_timestamp",
        "is_eligible_flag",
        "min_units_required",
        "discounted_quantity",
        "percentage_discount_rate"
    )
)

print(f"Datensätze vor Bereinigung: {bronze_df.count()} ")
print(f"Datensätze nach Transformation (ohne Duplikate): {silver_df.count()}")

display(silver_df.limit(5))





StatementMeta(, f1a4af41-4177-4af9-ad31-ab28ba48d7b4, 9, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 423 
Datensätze nach Transformation (ohne Duplikate): 423


SynapseWidget(Synapse.DataFrame, 2dd5ba2b-2e08-4d80-9d99-7bf13109d4cc)

# Dokumentation: Datenqualität & Null-Wert-Behandlung (active_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Lückenlose Absicherung der transformierten Marketing- und Rabattdaten gegen `NULL`- und `NaN`-Werte. Dieser Schritt garantiert eine mathematisch absolut stabile Basis für nachgelagerte Umsatz- und Effizienzanalysen im Gold Layer.

---

## 1. Pflichtspalten (Harter Filter)
Diese Spalten bilden das logische Fundament jeder Promotion. Datensätze, die diese Kriterien verletzen, werden rigoros aus dem Datenfluss entfernt, da sie analytisch nicht zugeordnet werden können.

* **`promotion_id`**: Ohne eine gültige Kampagnen-ID kann keine Zuordnung zu Marketing-Aktivitäten erfolgen.
* **`product_id`**: Ohne die Artikel-ID (EAN) ist es unmöglich zu bestimmen, welches Produkt rabattiert wurde.

---

## 2. Text-Standardisierung (Business-Platzhalter)
Unter Verwendung der zentralen Funktion `clean_string_nulls` werden alle Textfelder systematisch bereinigt. Technische Fragmente wie `"null"`, `"NaN"`, `"nan"` oder komplett leere Zeichenketten `""` werden durch vordefinierte Standardwerte ersetzt:

* **`customer_id`**: Wenn keine Kunden-ID übergeben wurde (z. B. bei anonymen Einkäufen ohne Kundenkarte an der Kasse), wird das Feld standardmäßig auf **`"GAST"`** gesetzt.

---

## 3. Numerischer Schutz & Flags (Schutz vor NULL und NaN)
Mengen, Bedingungen und Rabattsätze werden durch die Kombination aus `coalesce()` und `nanvl()` sowohl gegen klassische Datenbank-`NULL`-Werte als auch gegen mathematische `NaN`-Laufzeitfehler abgesichert:

* **`is_eligible_flag`**: Fehlt der Berechtigungsstatus, wird er standardmäßig auf **`0`** (Nicht berechtigt) gesetzt.
* **`min_units_required`**: Wenn der Schwellenwert für die Mindestmenge fehlt, wird er auf den neutralen Standardwert **`1`** gesetzt.
* **`discounted_quantity`**: Fehlt die Angabe zur rabattierten Menge, wird sie auf **`0`** gesetzt.
* **`percentage_discount_rate`**: Der prozentuale Rabattsatz wird bei Fehlern mathematisch auf **`0.0000`** (0% Rabatt) gezwungen. Dies verhindert fatale `Null-Pointer-Exceptions` bei der späteren Umsatzberechnung im Gold Layer (`Menge * Preis * Rabatt`).

---

## 4. Zeitstempel-Erhalt
* **`campaign_start`**, **`campaign_end`**, **`redemption_timestamp`**: Diese Zeitstempel werden für zeitraumbezogene Validierungen im Gold Layer unverändert mitgeführt.


In [10]:
from pyspark.sql.functions import col, coalesce, lit, nanvl, when, trim 

# Eine wiederverwendbare Logik für alle Textspalten
def clean_string_nulls(column_name, default_text):
    return when(
        (col(column_name).isNull()) | 
        (trim(col(column_name)) == "NaN") | 
        (trim(col(column_name)) == "nan") | 
        (trim(col(column_name)) == "null") |
        (trim(col(column_name)) == ""), 
        lit(default_text)
    ).otherwise(trim(col(column_name)))

# Finale Qualitätsbereinigung für die Marketing-Aktionen
silver_promotions_final = (silver_df
    # 1. PFLICHTSPALTEN FILTERN: Ohne Kampagnen-ID oder Artikel-ID ist die Zeile ungültig
    .filter(
        col("promotion_id").isNotNull() & (trim(col("promotion_id")) != "NaN") & (trim(col("promotion_id")) != "nan") &
        col("product_id").isNotNull() & (trim(col("product_id")) != "NaN") & (trim(col("product_id")) != "nan")
    )
    
    # 2. FINALE NULL & NaN BEHANDLUNG FÜR TEXT UND ZAHLEN
    .select(
        col("promotion_id"),
        col("product_id"),
        
        # TEXT-SPALTEN: Schützt vor NULL, "NaN", "null" und Leerzeichen
        clean_string_nulls("customer_id", "GAST").alias("customer_id"),
        
        # NUMERISCHE SPALTEN / FLAGS: Schützt vor echtem NULL und mathematischem NaN
        coalesce(nanvl(col("is_eligible_flag"), lit(0)), lit(0)).cast("int").alias("is_eligible_flag"),
        coalesce(nanvl(col("min_units_required"), lit(1)), lit(1)).cast("int").alias("min_units_required"),
        coalesce(nanvl(col("discounted_quantity"), lit(0)), lit(0)).cast("int").alias("discounted_quantity"),
        
        # Hochpräziser Rabattsatz (Decimal): Schutz vor NULL und NaN
        coalesce(nanvl(col("percentage_discount_rate"), lit(0.0000)), lit(0.0000)).cast("decimal(5,4)").alias("percentage_discount_rate"),
        
        # Zeitstempel für Gültigkeiten und Einlösungen mitführen
        col("campaign_start"),
        col("campaign_end"),
        col("redemption_timestamp")
    )
)

# 3. In die finale Silver Delta-Tabelle schreiben
silver_promotions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.active_promotions")

print(f"Promotion-Daten erfolgreich im Silver Layer gespeichert! Anzahl Datensätze: {silver_promotions_final.count()}")
display(silver_promotions_final.limit(5))


StatementMeta(, f1a4af41-4177-4af9-ad31-ab28ba48d7b4, 12, Finished, Available, Finished, False)

Promotion-Daten erfolgreich im Silver Layer gespeichert! Anzahl Datensätze: 423


SynapseWidget(Synapse.DataFrame, b2c5956a-6809-4885-a2bd-0f106728fb89)

Wenn Sie diese Daten später im Gold Layer für eine **geografische Landkarte** (über latitude/longitude) nutzen oder Berichte nach state (Bundesland) gruppieren, wird Ihr Dashboard niemals abstürzen oder hässliche, leere Zeilen anzeigen. Alle Daten weisen nun eine garantierte 100%ige Struktur auf.


# Silver Layer – Sales Orders Cleaning

## Ziel

In diesem Schritt werden die Sales Orders aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Datentypen**, der **Bereinigung von Kundeninformationen**, der **Konvertierung des Bestelldatums** sowie der **Auswahl der relevanten Spalten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Order Number standardisieren

Die `order_number` wird in den Datentyp `Long` konvertiert:

* `order_number` → `Long`

Dadurch wird sichergestellt, dass die Bestellnummer in einem einheitlichen numerischen Datentyp vorliegt.

### 2. Customer ID bereinigen

Die `customer_id` wird zunächst von führenden und nachfolgenden Leerzeichen bereinigt und anschließend als String gespeichert:

* `customer_id` → `String`

Die Verwendung eines Strings ist für Kunden-IDs geeignet, da diese als Identifikatoren und nicht als numerische Werte behandelt werden.

### 3. Customer Name bereinigen

Der Kundenname wird mit `trim()` bereinigt und als String gespeichert:

* `customer_name` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende des Wertes entfernt.

### 4. Anzahl der Bestellpositionen standardisieren

Das Feld `number_of_line_items` wird in einen Integer-Datentyp konvertiert:

* `number_of_line_items` → `Integer`

Damit kann die Anzahl der Bestellpositionen später für numerische Analysen verwendet werden.

### 5. Bestelldatum standardisieren

Das ursprüngliche Feld `order_datetime` enthält einen Unix-Timestamp.

Dieser wird zunächst in einen numerischen `Long`-Wert konvertiert und anschließend in einen Spark Timestamp umgewandelt:

* `order_datetime` → `orderdate`
* Unix Timestamp → `Timestamp`

Dadurch steht das Bestelldatum in einem standardisierten Format für zeitbasierte Analysen zur Verfügung.

### 6. Relevante Spalten auswählen

Nach den Transformationen werden nur die für den Silver Layer benötigten Spalten übernommen:

* `order_number`
* `customer_id`
* `customer_name`
* `number_of_line_items`
* `orderdate`

Das ursprüngliche Feld `order_datetime` wird dadurch nicht weitergeführt.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(orders_silver.limit(5))


In [24]:
from pyspark.sql.functions import col, trim, from_unixtime
from pyspark.sql import functions as F

bronze_df = spark.table("Retailorg_bronze.sales_orders")
# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["order_number","order_datetime"])
orders_silver = ( df_deduplicated 
    .withColumn("order_number", 
        F.col("order_number")
        .cast("long")
    )
 

    .withColumn("customer_id",
         trim(F.col("customer_id"))
         .cast("string")
    )

    .withColumn("customer_name",
          trim(F.col("customer_name"))
          .cast("string"))

    .withColumn("number_of_line_items", 
           F.col("number_of_line_items")
           .cast("int"))

    .withColumn("orderdate", 
        from_unixtime(F.col("order_datetime")
        .cast("long"))
        .cast("timestamp")
    )
)

##

orders_silver = (orders_silver
              .select(
                   "order_number",
                   "customer_id",
                   "customer_name",
                   "number_of_line_items",
                   "orderdate"
               )
)

print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {orders_silver.count()}")

display(orders_silver.limit(5))



StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 26, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 4074 
Datensätze nach Bereinigung (ohne Duplikate): 4074


SynapseWidget(Synapse.DataFrame, 7bea731e-b9db-4c42-bc7d-0ece3ad3363c)

# Dokumentation: Datenqualität & Null-Wert-Behandlung (sales_orders)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Sicherung einer fehlerfreien und konsistenten Grundlage für Umsatz-, Warenkorb- und Kundenkanal-Analysen im Gold Layer durch die gezielte Beseitigung und Standardisierung von `NULL`-Werten.

---

### 1. Pflichtspalten (Harter Filter)
Diese Spalten dürfen **niemals** `NULL` oder leer sein. Datensätze, die diese Kriterien verletzen, werden rigoros aus dem Datenfluss entfernt.

* **`order_number`**: Ein Verkaufsauftrag ohne eindeutige Belegnummer ist kaufmännisch und analytisch ungültig.
* **`customer_id`**: Ohne eine gültige Kunden-ID ist ein späterer Join mit den Kundenstammdaten (`dim_customers`) unmöglich.

### 2. Zahlen absichern (Numerischer Schutz)
Fehlende numerische Werte werden durch mathematisch neutrale Standardwerte ersetzt, um Berechnungsfehler (z. B. `Null-Pointer-Exceptions`) bei Aggregationen zu verhindern.

* **`number_of_line_items`**: Wenn die Anzahl der Positionen im Auftrag `NULL` ist, wird sie standardmäßig auf **`0`** gesetzt. Dies sichert die Berechnung der durchschnittlichen Warenkorbgröße.

### 3. Texte standardisieren (Business-Platzhalter)
Leere Textfelder werden durch sprechende Platzhalter ersetzt, um unschöne Lücken oder kryptische `NULL`-Einträge in Berichtsfiltern und BI-Dashboards zu vermeiden.

* **`customer_name`**: Fehlt der Kundenname (z. B. durch technische Fehler bei Express-Checkouts im Onlineshop), wird er durch den Standardwert **`"Gastbesteller"`** oder **`"Unbekannter Kunde"`** ersetzt.

### 4. Zeitstempel-Validierung
* **`orderdate`**: Ein fehlendes Bestelldatum führt zu Fehlern in Zeitreihenanalysen. Falls dieser Wert `NULL` ist, wird er auf den Verarbeitungszeitstempel des Systems (`current_timestamp()`) gesetzt, um die zeitliche Einordnung zu gewährleisten.


In [11]:
from pyspark.sql.functions import col, coalesce, lit, current_timestamp, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem orders_silver DataFrame
silver_orders_final = (orders_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder customer_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("customer_id").isNotNull() & 
        (trim(col("customer_id")) != "") &
        (trim(col("customer_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), 
        clean_string_nulls("customer_id", "UNBEKANNT").alias("customer_id"),
        
        # Text standardisieren unter Verwendung der clean_string_nulls Funktion
        clean_string_nulls("customer_name", "Gastbesteller").alias("customer_name"),
        
        # Zahlen absichern (NULL durch 0 ersetzen)
        coalesce(col("number_of_line_items"), lit(0)).alias("number_of_line_items"),
        
        # Zeitstempel absichern (Falls NULL, aktuellen System-Zeitstempel nutzen)
        coalesce(col("orderdate"), current_timestamp()).alias("order_date")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_orders_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.sales_orders")

# Protokollierung der Datensätze
print(f"Datensätze in orders_silver: {orders_silver.count()}")
print(f"Datensätze nach finaler Qualitätsbereinigung: {silver_orders_final.count()}")

display(silver_orders_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 13, Finished, Available, Finished, False)

Datensätze in orders_silver: 4074
Datensätze nach finaler Qualitätsbereinigung: 4074


SynapseWidget(Synapse.DataFrame, a017fbb7-eac7-41ed-8fdf-c36cfbc92be3)

# Silver Layer – Sales Orders Cleaning

## Ziel

In diesem Schritt werden die Sales Orders aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Datentypen**, der **Bereinigung von Kundeninformationen**, der **Konvertierung des Bestelldatums** sowie der **Auswahl der relevanten Spalten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Order Number standardisieren

Die `order_number` wird in den Datentyp `Long` konvertiert:

* `order_number` → `Long`

Dadurch wird sichergestellt, dass die Bestellnummer in einem einheitlichen numerischen Datentyp vorliegt.

### 2. Customer ID bereinigen

Die `customer_id` wird zunächst von führenden und nachfolgenden Leerzeichen bereinigt und anschließend als String gespeichert:

* `customer_id` → `String`

Die Verwendung eines Strings ist für Kunden-IDs geeignet, da diese als Identifikatoren und nicht als numerische Werte behandelt werden.

### 3. Customer Name bereinigen

Der Kundenname wird mit `trim()` bereinigt und als String gespeichert:

* `customer_name` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende des Wertes entfernt.

### 4. Anzahl der Bestellpositionen standardisieren

Das Feld `number_of_line_items` wird in einen Integer-Datentyp konvertiert:

* `number_of_line_items` → `Integer`

Damit kann die Anzahl der Bestellpositionen später für numerische Analysen verwendet werden.

### 5. Bestelldatum standardisieren

Das ursprüngliche Feld `order_datetime` enthält einen Unix-Timestamp.

Dieser wird zunächst in einen numerischen `Long`-Wert konvertiert und anschließend in einen Spark Timestamp umgewandelt:

* `order_datetime` → `orderdate`
* Unix Timestamp → `Timestamp`

Dadurch steht das Bestelldatum in einem standardisierten Format für zeitbasierte Analysen zur Verfügung.

### 6. Relevante Spalten auswählen

Nach den Transformationen werden nur die für den Silver Layer benötigten Spalten übernommen:

* `order_number`
* `customer_id`
* `customer_name`
* `number_of_line_items`
* `orderdate`

Das ursprüngliche Feld `order_datetime` wird dadurch nicht weitergeführt.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(orders_silver.limit(5))


In [15]:
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F

items_silver = df_deduplicated.select(
    F.col("order_number"),
    explode(F.col("ordered_products")).alias("product")
)

items_silver = (items_silver

     .withColumn("order_number", 
         F.col("order_number")
         .cast("long")
     )

     .withColumn("product_id",
          F.col("product.id")
          .cast("string")
     )

     .withColumn("product_name",
           F.col("product.name")
           .cast("string")
     )

     .withColumn("currency",
           F.col("product.curr")
           .cast("string")
     )

     .withColumn("price",
           F.col("product.price")
           .cast("decimal(18,2)")
     )
 

      .withColumn("quantity", 
            F.col("product.qty")
            .cast("int")
      )
       

     .withColumn("unit", 
            F.col("product.unit")
            .cast("string")
     )
)


items_silver=items_silver.select(
                  "order_number",
                  "product_id",
                  "product_name",
                  "currency",
                  "price",
                  "quantity",
                  "unit"
)
display(items_silver.limit(5))

StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9dfcb959-70bf-4c29-a788-e36345682533)

# Dokumentation: Datenqualität & Null-Wert-Behandlung (sales_order_items)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Transformation von geschachtelten Bestelldaten (*Explode*) in ein flaches, konsistentes Positionsdaten-Modell. Es sichert die mathematische Korrektheit aller Einzelpositionen für spätere Umsatz- und Margenberechnungen im Gold Layer ab.

---

### 1. Pflichtspalten (Harter Filter)
Diese Spalten verknüpfen den Artikel mit dem Auftrag. Fehlen sie, ist die Tabellenzeile unbrauchbar und wird ausgefiltert.

* **`order_number`**: Ohne Zuordnung zu einem Hauptauftrag verliert die Position ihre Gültigkeit.
* **`product_id`**: Jede Positionszeile muss zwingend einem konkreten Produkt zugeordnet sein, um spätere Stammdaten-Joins zu ermöglichen.

### 2. Zahlen absichern (Numerischer Schutz)
Mengen und Preise dürfen niemals `NULL` sein, um Berechnungsfehler bei der Ermittlung des Gesamtumsatzes zu verhindern.

* **`quantity`**: Wenn die Verkaufsmenge `NULL` ist, wird sie standardmäßig auf **`0`** gesetzt.
* **`price`**: Wenn kein Einzelpreis übergeben wurde, wird er mit **`0.00`** initialisiert.

### 3. Texte standardisieren (Business-Platzhalter)
Die Funktion `clean_string_nulls` bereinigt Textspalten von technischen Fehlern wie `"null"` oder `"NaN"`.

* **`product_name`**: Fehlende Namen werden durch den Platzhalter **`"Unbekannter Artikel"`** ersetzt.
* **`currency`**: Fehlt die Währung, wird der Standardwert **`"EUR"`** eingetragen.
* **`unit`**: Eine fehlende Verkaufseinheit wird standardmäßig als **`"Stück"`** deklariert.


In [16]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F

# 1. Null-Wert-Behandlung basierend auf Ihrem items_silver DataFrame
silver_items_final = (items_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder product_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("product_id").isNotNull() & 
        (trim(col("product_id")) != "") &
        (trim(col("product_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Key
        clean_string_nulls("product_id", "UNBEKANNT").alias("product_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("product_name", "Unbekannter Artikel").alias("product_name"),
        clean_string_nulls("currency", "EUR").alias("currency"),
        clean_string_nulls("unit", "Stück").alias("order_unit"),
        
        # Zahlen & Preise absichern
        coalesce(col("price"), lit(0.00)).alias("unit_price"),
        coalesce(col("quantity"), lit(0)).alias("quantity_ordered")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_items_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.sales_order_items")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode: {items_silver.count()}")
print(f"Datensätze nach finaler Positionsbereinigung: {silver_items_final.count()}")

# Schema anzeigen & Vorschau laden
silver_items_final.printSchema()
display(silver_items_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 18, Finished, Available, Finished, False)

Datensätze nach Explode: 8137
Datensätze nach finaler Positionsbereinigung: 8137
root
 |-- sales_order_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- order_unit: string (nullable = true)
 |-- unit_price: double (nullable = false)
 |-- quantity_ordered: integer (nullable = false)



SynapseWidget(Synapse.DataFrame, 2f63c515-85dc-4b81-b769-13cf410c8c56)

# Silver Layer – Order Items Cleaning

## Ziel

In diesem Schritt werden die einzelnen Bestellpositionen aus dem Bronze Layer extrahiert und technisch standardisiert.

Die Produktinformationen liegen im Bronze Layer als verschachtelte Struktur innerhalb des Feldes `ordered_products` vor.

Der Schwerpunkt liegt daher auf der **Auflösung der verschachtelten Produktstruktur**, der **Zuordnung der Produkte zu den jeweiligen Bestellungen** sowie der **Standardisierung der Datentypen**.

## Durchgeführte Schritte

### 1. Bestellpositionen aus verschachtelter Struktur extrahieren

Das Feld `ordered_products` enthält mehrere Produkte innerhalb einer Bestellung.

Mit `explode()` wird jedes Produkt in eine eigene Zeile umgewandelt:

```python
items_silver = bronze_df.select(
    col("order_number"),
    explode(col("ordered_products")).alias("product")
)

Eine Bestellung
      │
      ├── Produkt 1
      ├── Produkt 2
      └── Produkt 3
            ↓
Drei einzelne Zeilen




# Silver Layer – Clicked Items Cleaning

## Ziel

In diesem Schritt werden die geklickten Produkte aus dem Bronze Layer extrahiert und technisch standardisiert.

Die Klickinformationen liegen im Bronze Layer als verschachtelte Struktur im Feld `clicked_items` vor.

Der Schwerpunkt liegt auf der **Auflösung der verschachtelten Klickdaten**, der **Zuordnung der Klicks zu den jeweiligen Bestellungen**, der **Extraktion von Produkt-ID und Klickwert** sowie der **Standardisierung der Datentypen**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Geklickte Produkte extrahieren

Das Feld `clicked_items` enthält die geklickten Produkte einer Bestellung.

Mit `explode()` wird jedes Element aus der verschachtelten Struktur in eine eigene Zeile umgewandelt:

```python
.select(
    F.col("order_number"),
    F.explode(F.col("clicked_items")).alias("clicked")
)


In [18]:
from pyspark.sql.functions import element_at
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F


clicked_silver = (
    df_deduplicated
    .select(
        F.col("order_number"),
        F.explode(F.col("clicked_items")).alias("clicked")
    )
    .withColumn(
        "order_number",
        F.col("order_number").cast("long")
    )
    .withColumn(
        "product_id",
        F.element_at(F.col("clicked"), 1).cast("string")
    )
    .withColumn(
        "click_value",
        F.element_at(F.col("clicked"), 2).cast("int")
    )
    .drop("clicked")
)

display(clicked_silver.limit(5))




StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec20c5a8-23e7-4dd2-ab0f-016b77091d04)

# Dokumentation: Datenqualität & Null-Wert-Behandlung (sales_clicked_items)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Strukturierung und Standardisierung von Tracking-Daten zu geklickten Artikeln (*Clicked Items*). Diese Tabelle verarbeitet Interaktionsdaten aus dem Onlineshop, um das Kundenverhalten (Klick-Häufigkeit und Relevanz) im Gold Layer absturzsicher analysieren zu können.

---

### 1. Pflichtspalten (Harter Filter)
Diese Felder verknüpfen das Klick-Ereignis mit dem Warenkorb-Prozess. Fehlen sie, verliert der Datensatz seinen analytischen Wert.

* **`order_number`**: Die Zuordnung zur entsprechenden Sitzung bzw. zum Kundenauftrag muss zwingend vorhanden sein.
* **`product_id`**: Ohne die eindeutige ID des geklickten Produkts ist der Datensatz für Sortimentsanalysen unbrauchbar.

### 2. Zahlen absichern (Numerischer Schutz)
* **`click_value`**: Gibt die Anzahl oder Intensität der Klicks an (z. B. Position in der Liste oder Klick-Zähler). Wenn dieser Wert `NULL` ist, wird er standardmäßig auf **`0`** gesetzt, um Aggregationsfehler (wie `SUM` oder `AVG`) zu verhindern.

### 3. Texte standardisieren (Business-Platzhalter)
Die Funktion `clean_string_nulls` bereinigt technische Behelfs-Strings aus dem Web-Tracking.

* **`product_id`**: Sollte die ID durch Tracking-Fehler leer sein, wird sie durch **`"UNBEKANNT"`** ersetzt (greift, sofern nicht bereits durch den harten Filter aussortiert).


In [20]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem clicked_silver DataFrame
silver_clicked_final = (clicked_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder product_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("product_id").isNotNull() & 
        (trim(col("product_id")) != "") &
        (trim(col("product_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Join-Key
        clean_string_nulls("product_id", "UNBEKANNT").alias("product_id"),
        
        # Zahlen absichern (NULL durch 0 ersetzen)
        coalesce(col("click_value"), lit(0)).alias("click_value")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_clicked_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.sales_clicked_items")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode/Element_at: {clicked_silver.count()}")
print(f"Datensätze nach finaler Klick-Bereinigung: {silver_clicked_final.count()}")

# Schema- und Ergebnisvorschau
silver_clicked_final.printSchema()
display(silver_clicked_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 22, Finished, Available, Finished, False)

Datensätze nach Explode/Element_at: 16333
Datensätze nach finaler Klick-Bereinigung: 16333
root
 |-- sales_order_id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- click_value: integer (nullable = false)



SynapseWidget(Synapse.DataFrame, 1dbd8017-6134-45b6-b742-b00dc3d38366)

# Dokumentation: Transformation & Datenbereinigung (sales_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Normalisierung und Aufbereitung der angewendeten Rabatt- und Aktionsdaten (*Promo Info*) pro Verkaufsauftrag. Das Skript löst verschachtelte Strukturen auf und überführt sie in ein flaches, relationales Tabellenmodell für präzise Kampagnenanalysen im Gold Layer.

---

### 1. Granularität und Struktur (Explode-Operation)
Die Rohdaten enthalten die Rabattinformationen als verschachteltes Array (`promo_info`) innerhalb des Auftrags. 
* **Logik:** Über eine `explode()`-Operation wird das Array aufgebrochen. Jede angewendete Werbeaktion pro Bestellung erhält dadurch eine eigene, eigenständige Zeile in der Tabelle `promo_silver`.

### 2. Technische Datentyp-Konvertierungen (Casting)
Um eine performante Weiterverarbeitung und standardisierte Joins zu gewährleisten, werden die Attribute aus der extrahierten Struktur in exakte Datentypen überführt:

* **`order_number`** → `long`: Eindeutige Zuweisung zum Hauptauftrag (Primär-/Fremdschlüssel).
* **`promo_id`** → `long`: Eindeutige Kennung der Marketing- oder Rabattkampagne.
* **`promo_discount`** → `double`: Der prozentuale oder absolute Rabattwert (z. B. `0.15` für 15% Nachlass).
* **`promo_item`** → `string`: Identifikator des Artikels oder der Warengruppe, für die der Rabatt gilt.
* **`promo_quantity`** → `int`: Die Anzahl der Artikelpositionen, die für diese Promotion qualifiziert waren.

### 3. Datenbereinigung und Optimierung
* **Struktur-Bereinigung:** Nach dem erfolgreichen Entschachteln und Verarbeiten aller Unterfelder wird das ursprüngliche, komplexe Objekt `promotion` mittels `.drop()` vollständig entfernt. Das hält den Speicherbedarf der Delta-Tabelle minimal.
* **Persistierung:** Die Bereinigungsstufe wird im Modus `overwrite` direkt in die Zielumgebung geschrieben, um sicherzustellen, dass bei jedem Pipeline-Lauf der aktuellste, konsistente Stand in der Delta-Tabelle `Retailorg_silver.sales_promotion` vorliegt.


In [25]:
from pyspark.sql.functions import element_at
from pyspark.sql.functions import col, explode
from pyspark.sql import functions as F

promo_silver = df_deduplicated.select(
    col("order_number"),
    explode(col("promo_info")).alias("promotion")
)



promo_silver = (promo_silver

    .withColumn("order_number",
       F.col("order_number")
       .cast("long")

    )

     .withColumn("promo_id",
          F.col("promotion.promo_id")
          .cast("long")
     )

     .withColumn("promo_discount",
          F.col("promotion.promo_disc")
          .cast("double")
     )

     .withColumn("promo_item",
          F.col("promotion.promo_item")
          .cast("string")
     )

     .withColumn("promo_quantity", 
           F.col("promotion.promo_qty")
           .cast("int")
     )
      .drop("promotion")
       
)

display(promo_silver.limit(5))



StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a850ba90-aa83-43c7-8377-b85863076b94)

# Dokumentation: Datenqualität & Null-Wert-Behandlung (sales_promotion)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Auflösung und Strukturierung der angewendeten Rabatt- und Aktionsdaten (*Promo Info*) pro Verkaufsauftrag. Diese Tabelle bereitet die Zuordnung von Marketingkampagnen zu den Bestellungen auf, um die Effektivität von Promotions im Gold Layer fehlerfrei zu analysieren.

---

### 1. Pflichtspalten (Harter Filter)
Diese Felder verknüpfen die Werbeaktion mit dem konkreten Kaufbeleg. Ohne diese IDs verliert der Datensatz seine Aussagekraft.

* **`order_number`**: Die Zuordnung zum Hauptauftrag ist zwingend erforderlich.
* **`promo_id`**: Ohne eine gültige Kampagnen-ID kann die Rabattierung keiner Marketingaktion zugeordnet werden.

### 2. Zahlen absichern (Numerischer Schutz)
Preisminderungen und Aktionsmengen müssen zwingend mathematisch abgesichert werden, um `Null-Pointer-Exceptions` bei statistischen Berechnungen zu vermeiden.

* **`promo_discount`**: Gibt den Rabattwert an (z. B. 0.15 für 15%). Wenn dieser Wert `NULL` ist, wird er auf **`0.00`** (kein Rabatt) gesetzt.
* **`promo_quantity`**: Gibt an, für wie viele Artikel der Rabatt galt. Ein `NULL`-Wert wird standardmäßig durch **`0`** ersetzt.

### 3. Texte standardisieren (Business-Platzhalter)
Die Textspalten werden über die Funktion `clean_string_nulls` standardisiert, um technische Fragmente zu eliminieren.

* **`promo_item`**: Identifiziert den betroffenen Artikel oder die Warengruppe innerhalb der Aktion. Wenn das Feld leer oder `NULL` ist, wird es durch den Platzhalter **`"Kein Artikel"`** oder **`"Allgemeiner Rabatt"`** ersetzt.


In [27]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem promo_silver DataFrame
silver_promo_final = (promo_silver
    # PFLICHTSPALTEN FILTERN: Zeilen ohne order_number oder promo_id entfernen
    .filter(
        col("order_number").isNotNull() & 
        col("promo_id").isNotNull()
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("order_number").alias("sales_order_id"), # Einheitlicher Name für den Join-Key
        col("promo_id").cast("string").alias("promotion_id"), # ID als standardisierter String-Key
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("promo_item", "Allgemeiner Rabatt").alias("promoted_item_id"),
        
        # Zahlen & Rabatte absichern
        coalesce(col("promo_discount"), lit(0.00)).alias("promotion_discount"),
        coalesce(col("promo_quantity"), lit(0)).alias("promotion_quantity")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_promo_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.sales_promotion")

# Protokollierung der Datensätze
print(f"Datensätze nach Explode: {promo_silver.count()}")
print(f"Datensätze nach finaler Promo-Bereinigung: {silver_promo_final.count()}")

# Schema- und Ergebnisvorschau
silver_promo_final.printSchema()
display(silver_promo_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 29, Finished, Available, Finished, False)

Datensätze nach Explode: 434
Datensätze nach finaler Promo-Bereinigung: 434
root
 |-- sales_order_id: long (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- promoted_item_id: string (nullable = true)
 |-- promotion_discount: double (nullable = false)
 |-- promotion_quantity: integer (nullable = false)



SynapseWidget(Synapse.DataFrame, 8cffcbf1-8c99-44df-821b-2b2767a6eb23)

# Silver Layer – Suppliers Cleaning

## Ziel

In diesem Schritt werden die Lieferantendaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Lieferanten- und Steueridentifikatoren**, der **Bereinigung der Adressdaten**, der **Korrektur der Postleitzahlen**, der **Standardisierung geografischer Koordinaten** sowie der **Konvertierung der Datentypen**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Supplier Identifiers standardisieren

Die Identifikatoren werden von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `SUPPLIER_ID` → `supplier_id` (`String`)
* `TAX_ID` → `tax_id` (`String`)

Die Speicherung als String stellt sicher, dass die Identifikatoren nicht als numerische Werte interpretiert werden.

### 2. Supplier Information bereinigen

Die grundlegenden Lieferanteninformationen werden bereinigt und als String gespeichert:

* `supplier_name` → `String`
* `state` → `String`
* `city` → `String`

Für die Textfelder wird `trim()` verwendet, um führende und nachfolgende Leerzeichen zu entfernen.

### 3. Postcode standardisieren

Die Postleitzahl wird zunächst von Leerzeichen bereinigt.

Zusätzlich wird ein eventuell vorhandener Dezimalzusatz `.0` am Ende entfernt:

```text
1938.0 → 1938


In [28]:
from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace
)
from pyspark.sql import functions as F


bronze_df = spark.table("Retailorg_bronze.suppliers")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["supplier_id"])
suppliers_silver = (df_deduplicated

    # Supplier identifiers
    .withColumn("supplier_id",
         trim(F.col("SUPPLIER_ID"))
         .cast("string")
     )


    .withColumn("tax_id",
         trim(F.col("TAX_ID"))
         .cast("string")
     )


    # Supplier information
    .withColumn("supplier_name",
          trim(F.col("supplier_name"))
           .cast("string")
     )

    .withColumn("state",
           trim(F.col("state"))
           .cast("string")
     )

    .withColumn("city",
            trim(F.col("city"))
            .cast("string")
     )

    # 1938.0 -> 1938
    .withColumn("postcode",
            regexp_replace(
            trim(F.col("postcode")),
             r"\.0$",
             ""
             )
             .cast("string")
     )

    # Address
    .withColumn("street",
            trim(F.col("street"))
            .cast("string")
     )

    .withColumn("house_number",
             trim(F.col("number"))
             .cast("int")
     )

    .withColumn("unit",
             trim(F.col("unit"))
             .cast("int")
     )

    .withColumn("region",
              trim(col("region"))
              .cast("string")
     )

    .withColumn("district",
               trim(F.col("district"))
               .cast("string")
     )

    # Coordinates
    .withColumn("longitude",
                trim(F.col("lon"))
                .cast("double")
     )

    .withColumn("latitude",
                 trim(F.col("lat"))
                 .cast("double")
     )
   

    # Products supplied
    .withColumn("items_provided",
                  trim(F.col("items_provided"))
                  .cast("string")
     )
    .drop("lat", "lon","number")
)


print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {suppliers_silver.count()}")
display(suppliers_silver.limit(5))
###############



StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 30, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 15 
Datensätze nach Bereinigung (ohne Duplikate): 15


SynapseWidget(Synapse.DataFrame, 6c9fad93-b2ac-4704-9b70-b1b3a1b5728e)

# Dokumentation: Transformation & Datenbereinigung (suppliers)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Bereinigung der Lieferanten-Stammdaten (*Suppliers*). Das Skript überführt unstrukturierte Adress- und Geodaten in ein konsistentes, relationales Format, um fehlerfreie Berichte und geografische Analysen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["supplier_id"])` wird sichergestellt, dass jeder Lieferant genau einmal in der Stammdatentabelle existiert. Redundante Datensätze werden direkt eliminiert.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **`supplier_id` & `tax_id`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`.
* **`postcode`** → `string`: Entfernung von technischen Konvertierungsartefakten (z. B. Umwandlung von Fließkommazahlen wie `1938.0` in saubere Strings wie `1938`) mittels `regexp_replace`.
* **`house_number` & `unit`** → `int`: Transformation der Adresskomponenten in numerische Ganzzahlen für standardisierte Abfragen.
* **`longitude` & `latitude`** → `double`: Konvertierung der geografischen Koordinaten in hochpräzise Fließkommazahlen für Mapping- und Logistik-Dashboards.

### 3. Struktur-Optimierung & Speicherbereinigung
* **Spalten-Bereinigung:** Die ursprünglichen, unbereinigten Quellspalten (`lat`, `lon`, `number`) werden nach der erfolgreichen Transformation mittels `.drop()` entfernt, um den Speicherbedarf im Delta Lake minimal zu halten.
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.suppliers` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [29]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F



# 1. Null-Wert-Behandlung basierend auf Ihrem suppliers_silver DataFrame
silver_suppliers_final = (suppliers_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne supplier_id rigoros herausfiltern
    .filter(
        col("supplier_id").isNotNull() & 
        (trim(col("supplier_id")) != "") &
        (trim(col("supplier_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("supplier_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("supplier_name", "Unbekannter Lieferant").alias("supplier_name"),
        clean_string_nulls("tax_id", "Nicht hinterlegt").alias("tax_id"),
        clean_string_nulls("state", "Unbekannt").alias("state"),
        clean_string_nulls("city", "Unbekannte Stadt").alias("city"),
        clean_string_nulls("postcode", "00000").alias("postcode"),
        clean_string_nulls("street", "Unbekannte Straße").alias("street"),
        clean_string_nulls("region", "Keine Region").alias("region"),
        clean_string_nulls("district", "Kein Bezirk").alias("district"),
        clean_string_nulls("items_provided", "Keine Angabe").alias("items_provided"),
        
        # Zahlen absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("house_number"), lit(0)).alias("house_number"),
        coalesce(col("unit"), lit(0)).alias("unit"),
        coalesce(col("longitude"), lit(0.00)).alias("longitude"),
        coalesce(col("latitude"), lit(0.00)).alias("latitude")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_suppliers_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.suppliers")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {suppliers_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_suppliers_final.count()}")

# Schema- und Ergebnisvorschau
silver_suppliers_final.printSchema()
display(silver_suppliers_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 31, Finished, Available, Finished, False)

Datensätze nach erster Bereinigung: 15
Datensätze nach finaler Qualitäts-Sicherung: 15
root
 |-- supplier_id: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- items_provided: string (nullable = true)
 |-- house_number: integer (nullable = false)
 |-- unit: integer (nullable = false)
 |-- longitude: double (nullable = false)
 |-- latitude: double (nullable = false)



SynapseWidget(Synapse.DataFrame, ef74b643-bbc7-4ed4-965d-eba7f82133dc)

# Silver Layer – Promotions Cleaning

## Ziel

In diesem Schritt werden die Promotions aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Promotion-Informationen**, der **Konvertierung der Rabattwerte und Mengenangaben**, der **Bereinigung der Produktreferenzen** sowie der **Standardisierung des Gültigkeitszeitraums**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Promotion ID standardisieren

Die `promotion_id` wird von führenden und nachfolgenden Leerzeichen bereinigt und als String gespeichert:

* `promotion_id` → `String`

Die Promotion-ID wird als String behandelt, da es sich um einen Identifikator handelt.

### 2. Promotion Type bereinigen

Das Feld `promotion_type` wird mit `trim()` bereinigt und als String gespeichert:

* `promotion_type` → `String`

Damit liegt der Promotion-Typ in einem einheitlichen Format vor.

### 3. Dollar Discount standardisieren

Der absolute Rabattbetrag wird bereinigt und als Dezimalwert mit zwei Nachkommastellen gespeichert:

* `dollar_discount` → `Decimal(18,2)`

Dadurch kann der Rabatt später zuverlässig für finanzielle Berechnungen verwendet werden.

### 4. Percent Discount standardisieren

Der prozentuale Rabatt wird als Dezimalwert gespeichert:

* `percent_discount` → `Decimal(5,4)`

Die Werte werden als Dezimalanteil interpretiert.

Beispiel:

```text
0.03 → 3 %
0.15 → 15 %


In [30]:
from pyspark.sql import functions as F

bronze_df = spark.table("Retailorg_bronze.promotions")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["promotion_id"])
promotions_silver = (
    df_deduplicated

    # Promotion ID
    .withColumn(
        "promotion_id",
        F.trim(F.col("promotion_id")).cast("string")
    )

    # Promotion type
    .withColumn(
        "promotion_type",
        F.trim(F.col("promotion_type")).cast("string")
    )

    # Dollar discount
    .withColumn(
        "dollar_discount",
        F.trim(F.col("dollar_discount")).cast("decimal(18,2)")
    )

    # Percent discount
    # Example: 0.03 = 3%
    .withColumn(
        "percent_discount",
        F.trim(F.col("percent_discount")).cast("decimal(5,4)")
    )

    # Product IDs
    .withColumn(
        "qualifying_products",
        F.trim(F.col("qualifying_products")).cast("string")
    )

    # Required units
    .withColumn(
        "units_required",
        F.trim(F.col("units_required")).cast("int")
    )

    # Free products
    .withColumn(
        "free_product_ids",
        F.trim(F.col("free_product_ids")).cast("string")
    )

    # Promotion length
    .withColumn(
        "length",
        F.trim(F.col("length")).cast("int")
    )

    # Timestamps
    .withColumn(
        "valid_from",
        F.to_timestamp(F.trim(F.col("valid_from")))
    )
    .withColumn(
        "valid_to",
        F.to_timestamp(F.trim(F.col("valid_to")))
    )
)
print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {promotions_silver.count()}")

display(promotions_silver.limit(5))



StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 32, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 4 
Datensätze nach Bereinigung (ohne Duplikate): 4


SynapseWidget(Synapse.DataFrame, 132b793e-2820-48dd-862b-2f51087a9eda)

# Dokumentation: Transformation & Datenbereinigung (promotions)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Bereinigung der Marketing- und Kampagnenstammdaten (*Promotions*). Das Skript überführt unstrukturierte Rabatt- und Laufzeitangaben in ein konsistentes, relationales Format, um fehlerfreie Deckungsbeitrags- und Kampagnenanalysen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["promotion_id"])` wird sichergestellt, dass jede Werbeaktion genau einmal in der Stammdatentabelle existiert. Redundante oder mehrfach übertragene Aktionscodes werden direkt eliminiert.

### 2. Technische Datentyp-Konvertierungen & Präzision
* **`promotion_id` & `promotion_type`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`.
* **`dollar_discount`** → `decimal(18,2)`: Festlegung einer exakten Festkommazahl für absolute Geldrabatte, um Rundungsfehler bei Finanzberechnungen zu verhindern.
* **`percent_discount`** → `decimal(5,4)`: Hochpräziser Datentyp für prozentuale Nachlässe (z. B. speichert `0.0300` exakt 3% ab).
* **`units_required` & `length`** → `int`: Transformation von Mindestbestellmengen und Kampagnenlaufzeiten in Ganzzahlen für mathematische Schwellenwertprüfungen.
* **`valid_from` & `valid_to`** → `timestamp`: Konvertierung der Gültigkeitszeiträume in echte Zeitstempel für präzise, zeitraumbezogene Joins mit den Verkaufstransaktionen.

### 3. Datenoptimierung & Persistierung
* **Struktur-Erhalt:** Alle Listen- und Produktzuordnungen (`qualifying_products`, `free_product_ids`) werden für spätere Filterprozesse als saubere Zeichenketten mitgeführt.
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.promotions` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [32]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem promotions_silver DataFrame
silver_promotions_final = (promotions_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne promotion_id rigoros herausfiltern
    .filter(
        col("promotion_id").isNotNull() & 
        (trim(col("promotion_id")) != "") &
        (trim(col("promotion_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("promotion_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("promotion_type", "Standard-Rabatt").alias("promotion_type"),
        clean_string_nulls("qualifying_products", "Alle").alias("qualifying_products"),
        clean_string_nulls("free_product_ids", "Keine").alias("free_product_ids"),
        
        # Zahlen & Rabatte absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("dollar_discount"), lit(0.00)).alias("dollar_discount"),
        coalesce(col("percent_discount"), lit(0.0000)).alias("percent_discount"),
        coalesce(col("units_required"), lit(1)).alias("units_required"), # Standardmäßig mindestens 1 Stück benötigt
        coalesce(col("length"), lit(0)).alias("campaign_length_days"),
        
        # Zeitstempel unverändert übernehmen (können im Gold Layer für Zeitfenster-Prüfungen genutzt werden)
        col("valid_from"),
        col("valid_to")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_promotions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.promotions")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {promotions_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_promotions_final.count()}")

# Schema- und Ergebnisvorschau
silver_promotions_final.printSchema()
display(silver_promotions_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 34, Finished, Available, Finished, False)

Datensätze nach erster Bereinigung: 4
Datensätze nach finaler Qualitäts-Sicherung: 4
root
 |-- promotion_id: string (nullable = true)
 |-- promotion_type: string (nullable = true)
 |-- qualifying_products: string (nullable = true)
 |-- free_product_ids: string (nullable = true)
 |-- dollar_discount: double (nullable = false)
 |-- percent_discount: double (nullable = false)
 |-- units_required: integer (nullable = false)
 |-- campaign_length_days: integer (nullable = false)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, f0257df7-163b-426f-8b38-5ce0e34e8165)

# Silver Layer – Products Cleaning

## Ziel

In diesem Schritt werden die Produktdaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Produktidentifikatoren**, der **Bereinigung von Produktinformationen**, der **Aufbereitung des Verkaufspreises** sowie der **Standardisierung der EAN- und Einheitendaten**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Product Identifier standardisieren

Die Produkt-ID wird von führenden und nachfolgenden Leerzeichen bereinigt und als String gespeichert:

* `product_id` → `String`

Die Speicherung als String stellt sicher, dass die Produkt-ID als Identifikator und nicht als numerischer Wert behandelt wird.

### 2. Product Category bereinigen

Die Produktkategorie wird mit `trim()` bereinigt und als String gespeichert:

* `product_category` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende entfernt.

### 3. Product Name bereinigen

Der Produktname wird ebenfalls bereinigt und als String gespeichert:

* `product_name` → `String`

Die Verwendung von `trim()` sorgt für eine einheitliche Darstellung der Produktnamen.

### 4. Sales Price bereinigen und standardisieren

Das Feld `sales_price` kann beispielsweise mit Währungs- oder Tausendertrennzeichen vorliegen.

Mit `regexp_replace()` werden zunächst `$`- und `,`-Zeichen entfernt:

```text
$1,299.99 → 1299.99


In [34]:
from pyspark.sql.functions import col, trim, regexp_replace
from pyspark.sql import functions as F

# Bronze-Tabelle laden
bronze_df = spark.table("Retailorg_bronze.products")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["product_id"])

# Bronze -> Silver
products_silver = (df_deduplicated

    # Product identifier
    .withColumn("product_id",
            trim(F.col("product_id"))
            .cast("string")
    )


    # Product category
    .withColumn("product_category",
             trim(F.col("product_category"))
             .cast("string")
    )


    # Product name
    .withColumn("product_name",
             trim(F.col("product_name"))
        .cast("string")
    )
  

    # Sales price
    .withColumn("sales_price",
              regexp_replace(
              trim(F.col("sales_price")),
              r"[$,]",
              ""
              )
              .cast("decimal(18,2)")
    )
 

    # EAN identifiers
    .withColumn("ean13",
              trim(F.col("EAN13"))
              .cast("string")
    )

    .withColumn("ean5",
              trim(F.col("EAN5"))
              .cast("string")
    )

    # Product unit
    .withColumn("product_unit",
               trim(F.col("product_unit"))
               .cast("string")
    )

)
print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {products_silver.count()}")

display(products_silver.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 36, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 98 
Datensätze nach Bereinigung (ohne Duplikate): 98


SynapseWidget(Synapse.DataFrame, 33427706-4fea-4d5c-8ee8-fb86bb8bce67)

# Dokumentation: Transformation & Datenbereinigung (products)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Strukturierung des zentralen Artikel- und Produktkatalogs (*Products*). Das Skript konsolidiert Stammdaten, reinigt Preiszeichen und bereitet internationale Barcodes (EAN) vor, um konsistente Produkt-Joins und fehlerfreie Umsatzanalysen im Gold Layer zu ermöglichen.

---

### 1. Eindeutigkeit & Datenkonsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["product_id"])` wird sichergestellt, dass jeder Artikel genau einmal in der Stammdatentabelle existiert. Redundante oder doppelt eingepflegte Produkt-IDs werden eliminiert.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **`product_id`, `product_name`, `product_category` & `product_unit`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`, um fehlerhafte Zeichenketten-Vergleiche auszuschließen.
* **`sales_price`** → `decimal(18,2)`: Entfernung von Währungssymbolen (z. B. `$`) und Formatierungszeichen mittels `regexp_replace`. Die Konvertierung in ein exaktes Festkomma-Format schützt vor Rundungsfehlern bei Margenberechnungen.
* **`ean13` & `ean5`** → `string`: Standardisierung der Barcodes als Textfelder, um führende Nullen (z. B. bei internationalen Artikelnummern) im Speicher vollständig zu erhalten.

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.products` geschrieben. Dies stellt sicher, dass nachgelagerte Transaktionstabellen stets gegen ein sauberes und aktuelles Artikelregister validiert werden.


In [38]:
from pyspark.sql.functions import col, coalesce, lit, trim, when
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem products_silver DataFrame
silver_products_final = (products_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne Produkt-ID oder ohne EAN13 rigoros herausfiltern
    .filter(
        col("product_id").isNotNull() & (trim(col("product_id")) != "") & (trim(col("product_id")) != "null") &
        col("ean13").isNotNull() & (trim(col("ean13")) != "") & (trim(col("ean13")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("product_id"),
        col("ean13"),
        
        # Zusatz-Barcode absichern (Wenn leer, dann leerer String statt NULL)
        clean_string_nulls("ean5", "").alias("ean5"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("product_name", "Unbekannter Artikel").alias("product_name"),
        clean_string_nulls("product_category", "Sonstiges").alias("product_category"),
        clean_string_nulls("product_unit", "Stück").alias("product_unit"),
        
        # Standard-Verkaufspreis absichern (Ersetzen durch 0.00, falls NULL)
        coalesce(col("sales_price"), lit(0.00)).alias("sales_price")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_products_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.products")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {products_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_products_final.count()}")

# Schema- und Ergebnisvorschau
silver_products_final.printSchema()
display(silver_products_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 40, Finished, Available, Finished, False)

Datensätze nach erster Bereinigung: 98
Datensätze nach finaler Qualitäts-Sicherung: 98
root
 |-- product_id: string (nullable = true)
 |-- ean13: string (nullable = true)
 |-- ean5: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_unit: string (nullable = true)
 |-- sales_price: double (nullable = false)



SynapseWidget(Synapse.DataFrame, fd1654f8-b91d-4652-96d7-07c76ee89728)

# Silver Layer – Loyalty Segments Cleaning

## Ziel

In diesem Schritt werden die Loyalty-Segment-Daten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Segment-ID**, der **Bereinigung der Segmentbeschreibung**, der **Konvertierung des Mengenschwellwerts** sowie der **Standardisierung des Gültigkeitszeitraums**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Loyalty Segment ID standardisieren

Die `loyalty_segment_id` wird zunächst von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `loyalty_segment_id` → `String`

Die Segment-ID wird als String behandelt, da es sich um einen Identifikator handelt.

### 2. Loyalty Segment Description bereinigen

Die Beschreibung des Loyalty-Segments wird mit `trim()` bereinigt und als String gespeichert:

* `loyalty_segment_description` → `String`

Dadurch werden führende und nachfolgende Leerzeichen entfernt und eine einheitliche Darstellung der Beschreibung sichergestellt.

### 3. Unit Threshold standardisieren

Das Feld `unit_threshold` beschreibt den Mengenschwellwert, ab dem ein Loyalty-Segment angewendet werden kann.

Der Wert wird bereinigt und in einen Integer-Datentyp konvertiert:

* `unit_threshold` → `Integer`

Dadurch kann der Schwellenwert später für numerische Vergleiche und Berechnungen verwendet werden.

### 4. Gültigkeitszeitraum standardisieren

Die Felder `valid_from` und `valid_to` definieren den Zeitraum, in dem das Loyalty-Segment gültig ist.

Beide Felder werden zunächst von überflüssigen Leerzeichen bereinigt und anschließend in einen Timestamp konvertiert:

* `valid_from` → `Timestamp`
* `valid_to` → `Timestamp`

Dadurch können die Loyalty-Segmente später anhand ihres Gültigkeitszeitraums analysiert und zeitliche Bedingungen geprüft werden.

## Datenqualität überprüfen

Nach der Transformation werden die ersten fünf Datensätze des bereinigten DataFrames angezeigt:

```python
display(loyalty_segments_silver.limit(5))


In [21]:
from pyspark.sql.functions import col, trim, to_timestamp
from pyspark.sql import functions as F

# Bronze-Tabelle laden
bronze_df = spark.table("Retailorg_bronze.loyalty_segments")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["loyalty_segment_id"])


# Bronze -> Silver
loyalty_segments_silver = (bronze_df

    # Loyalty segment ID
    .withColumn("loyalty_segment_id",
           trim(F.col("loyalty_segment_id")
           .cast("string") 
           )
    )

    # Description
    .withColumn("loyalty_segment_description",
            trim(F.col("loyalty_segment_description"))
            .cast("string")
    )
  

    # Unit threshold
    .withColumn("unit_threshold",
             trim(F.col("unit_threshold"))
             .cast("int")
    )
   

    # Validity
    .withColumn("valid_from",
              to_timestamp(
              trim(F.col("valid_from"))
             )
    )

    .withColumn("valid_to",
              to_timestamp(
              trim(F.col("valid_to"))
              )
         )
)
print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {loyalty_segments_silver.count()}")

display(loyalty_segments_silver.limit(5))

loyalty_segments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.loyalty_segments")

StatementMeta(, 082368ab-d3b3-4c3e-9003-dfd433bdbf44, 23, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 4 
Datensätze nach Bereinigung (ohne Duplikate): 4


SynapseWidget(Synapse.DataFrame, d77ea98e-8bcc-4f29-8c29-3e069c55b3e8)

# Silver Layer – Company Employees Cleaning

## Ziel

In diesem Schritt werden die Mitarbeiterdaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Bereinigung und Standardisierung der Mitarbeiterinformationen**, der **Umwandlung des Aktivitätsstatus in einen Boolean-Wert** sowie der **Standardisierung des Gültigkeitszeitraums der Mitarbeiterdatensätze**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Employee ID standardisieren

Die `employee_id` wird von führenden und nachfolgenden Leerzeichen bereinigt und als String gespeichert:

* `employee_id` → `String`

Die Mitarbeiter-ID wird als String behandelt, da es sich um einen Identifikator handelt.

### 2. Employee Name bereinigen

Der Mitarbeitername wird mit `trim()` bereinigt und als String gespeichert:

* `employee_name` → `String`

Dadurch werden überflüssige Leerzeichen am Anfang und Ende des Namens entfernt.

### 3. Department standardisieren

Das Feld `department` wird bereinigt und als String gespeichert:

* `department` → `String`

Damit liegt die Abteilungsinformation in einem einheitlichen Format vor.

### 4. Region standardisieren

Die Region wird von überflüssigen Leerzeichen bereinigt und als String gespeichert:

* `region` → `String`

### 5. Employee Key standardisieren

Der `employee_key` wird bereinigt und als String gespeichert:

* `employee_key` → `String`

Der Employee Key dient als technischer Identifikator für den Mitarbeiterdatensatz.

### 6. Active Record standardisieren

Das Feld `active_record` enthält einen Status, der ursprünglich als `1` oder `0` gespeichert ist.

Dieser Status wird in einen Boolean-Wert konvertiert:

* `1` → `True`
* `0` → `False`
* andere Werte → `NULL`

Dadurch wird der Aktivitätsstatus technisch standardisiert und kann später direkt als Boolean-Bedingung verwendet werden.

### 7. Start des aktiven Datensatzes

Das Feld `active_record_start` beschreibt den Beginn der Gültigkeit eines Mitarbeiterdatensatzes.

Der Wert wird zunächst bereinigt und anschließend in einen Timestamp konvertiert:

* `active_record_start` → `Timestamp`

Dadurch kann der Beginn der Gültigkeit für zeitbasierte Analysen und Abfragen verwendet werden.

### 8. Ende des aktiven Datensatzes

Das Feld `active_record_end` beschreibt das Ende der Gültigkeit eines Mitarbeiterdatensatzes.

Auch dieses Feld wird in einen Timestamp konvertiert:

* `active_record_end` → `Timestamp`

Damit kann der vollständige Gültigkeitszeitraum eines Mitarbeiterdatensatzes abgebildet werden.

## Datenqualität überprüfen

Die Transformation verwendet eine explizite Spaltenauswahl mit `select()`.

Dadurch werden ausschließlich die benötigten und bereits transformierten Spalten in den Silver DataFrame übernommen.

Besonders relevant für die Datenqualität sind:

* korrekt formatierte Mitarbeiter-IDs
* bereinigte Mitarbeiternamen
* standardisierte Abteilungen und Regionen
* korrekte Employee Keys
* Boolean-Werte für den Aktivitätsstatus
* korrekt konvertierte Start- und Endzeitpunkte

## Speicherung im Silver Layer

Der bereinigte DataFrame wird als Delta Table gespeichert:

```python
company_employees_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.company_employees")


In [39]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    to_timestamp
)

bronze_df = spark.table("Retailorg_bronze.company_employees")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["employee_id"])



company_employees_silver = df_deduplicated.select(

    # Employee ID
    trim(col("employee_id"))
        .cast("string")
        .alias("employee_id"),

    # Employee name
    trim(col("employee_name"))
        .cast("string")
        .alias("employee_name"),

    # Department
    trim(col("department"))
        .cast("string")
        .alias("department"),

    # Region
    trim(col("region"))
        .cast("string")
        .alias("region"),

    # Employee key
    trim(col("employee_key"))
        .cast("string")
        .alias("employee_key"),

    # 1 = active, 0 = inactive
    when(trim(col("active_record")) == "1", True)
        .when(trim(col("active_record")) == "0", False)
        .otherwise(None)
        .alias("active_record"),

    # Start of active record
    to_timestamp(
        trim(col("active_record_start"))
    ).alias("active_record_start"),

    # End of active record
    to_timestamp(
        trim(col("active_record_end"))
    ).alias("active_record_end")
)


print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {company_employees_silver.count()}")

display(company_employees_silver.limit(5))




StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 41, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 510 
Datensätze nach Bereinigung (ohne Duplikate): 500


SynapseWidget(Synapse.DataFrame, 9c179a95-3d28-4f56-8b3d-0a49e5956a16)

# Dokumentation: Transformation & Datenbereinigung (company_employees)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung und Historisierung der Mitarbeiter-Stammdaten (*Company Employees*). Das Skript überführt unstrukturierte Mitarbeiterdaten und Gültigkeitszeiträume in ein konsistentes, relationales SCD-Format (Slowly Changing Dimensions), um fehlerfreie Zuordnungen von Personalressourcen zu Verkaufsbelegen oder Einkaufsprozessen im Gold Layer zu gewährleisten.

---

### 1. Eindeutigkeit & Konsistenz (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["employee_id"])` wird sichergestellt, dass jeder Mitarbeiter pro Schlüssel genau einmal in der Verarbeitung berücksichtigt wird. Redundante Datensätze werden eliminiert.

### 2. Technische Datentyp-Konvertierungen & Flag-Logik
* **`employee_id`, `employee_name`, `department`, `region` & `employee_key`** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()`, um systemübergreifende Joins abzusichern.
* **`active_record`** → `boolean`: Transformation des rohen Text-Flags (`"1"` / `"0"`) in einen nativen Booleschen Wert (`True` / `False`), was die Performance bei Filterungen optimiert.
* **`active_record_start` & `active_record_end`** → `timestamp`: Konvertierung der Datumsangaben in echte Zeitstempel für lückenlose Zeitreihen- und Gültigkeitsprüfungen.

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.company_employees` geschrieben, um eine saubere, stets aktuelle Datengrundlage bereitzustellen.


In [41]:
from pyspark.sql.functions import col, coalesce, lit, trim, when, current_timestamp
from pyspark.sql import functions as F


# 1. Null-Wert-Behandlung basierend auf Ihrem company_employees_silver DataFrame
silver_employees_final = (company_employees_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne employee_id rigoros herausfiltern
    .filter(
        col("employee_id").isNotNull() & 
        (trim(col("employee_id")) != "") & 
        (trim(col("employee_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("employee_id"),
        col("employee_key"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("employee_name", "Unbekannter Mitarbeiter").alias("employee_name"),
        clean_string_nulls("department", "Ohne Zuordnung").alias("department"),
        clean_string_nulls("region", "Unbekannte Region").alias("region"),
        
        # Boolean-Flag absichern (Standardmäßig im Zweifel auf False setzen)
        coalesce(col("active_record"), lit(False)).alias("active_record"),
        
        # Zeitstempel absichern (Start auf aktuellen Systemzeitstempel, Ende auf unendlich/Standard-Max)
        coalesce(col("active_record_start"), current_timestamp()).alias("active_record_start"),
        coalesce(col("active_record_end"), F.to_timestamp(lit("9999-12-31 23:59:59"))).alias("active_record_end")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_employees_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.company_employees")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {company_employees_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_employees_final.count()}")

# Schema- und Ergebnisvorschau
silver_employees_final.printSchema()
display(silver_employees_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 43, Finished, Available, Finished, False)

Datensätze nach erster Bereinigung: 500
Datensätze nach finaler Qualitäts-Sicherung: 500
root
 |-- employee_id: string (nullable = true)
 |-- employee_key: string (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- region: string (nullable = true)
 |-- active_record: boolean (nullable = false)
 |-- active_record_start: timestamp (nullable = false)
 |-- active_record_end: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 714129e6-2f57-4986-ab65-c6ee12423bf1)

# Silver Layer – Customers Cleaning

## Ziel

In diesem Schritt werden die Kundendaten aus dem Bronze Layer bereinigt und technisch standardisiert.

Der Schwerpunkt liegt auf der **Standardisierung der Kunden- und Steueridentifikatoren**, der **Bereinigung der Kunden- und Adressinformationen**, der **Korrektur von Postleitzahlen und Hausnummern**, der **Konvertierung geografischer Koordinaten und Unix-Timestamps** sowie der **Standardisierung der gekauften Einheiten und Loyalty-Segmente**.

Anschließend werden die bereinigten Daten als **Delta Table im Silver Layer** gespeichert.

## Durchgeführte Schritte

### 1. Customer Identifiers standardisieren

Die Kunden- und Steueridentifikatoren werden mit `trim()` bereinigt und als String gespeichert:

* `customer_id` → `String`
* `tax_id` → `String`
* `tax_code` → `String`

Die Speicherung als String stellt sicher, dass Identifikatoren nicht als numerische Werte interpretiert werden.

### 2. Customer Information bereinigen

Die grundlegenden Kundeninformationen werden bereinigt und als String gespeichert:

* `customer_name` → `String`
* `state` → `String`
* `city` → `String`

Durch `trim()` werden führende und nachfolgende Leerzeichen entfernt.

### 3. Postcode standardisieren

Die Postleitzahl kann beispielsweise als numerischer Wert mit einem Dezimalzusatz vorliegen.

Der Dezimalzusatz `.0` wird entfernt:

```text
46506.0 → 46506


In [42]:
from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace,
    from_unixtime
)

# Bronze-Tabelle laden
bronze_df = spark.table("Retailorg_bronze.customers")

# Wir entfernen Zeilen, bei denen die kritischen IDs identisch sind
df_deduplicated = bronze_df.dropDuplicates(["customer_id", "valid_from"])


# Bronze -> Silver
customers_silver = df_deduplicated.select(

    # Customer identifiers
    trim(col("customer_id"))
        .cast("string")
        .alias("customer_id"),

    trim(col("tax_id"))
        .cast("string")
        .alias("tax_id"),

    trim(col("tax_code"))
        .cast("string")
        .alias("tax_code"),

    # Customer information
    trim(col("customer_name"))
        .cast("string")
        .alias("customer_name"),

    trim(col("state"))
        .cast("string")
        .alias("state"),

    trim(col("city"))
        .cast("string")
        .alias("city"),

    # Postcode: 46506.0 -> 46506
    regexp_replace(
        trim(col("postcode")),
        r"\.0$",
        ""
    ).alias("postcode"),

    # Address
    trim(col("street"))
        .cast("string")
        .alias("street"),

    # Number: 521.0 -> 521
    regexp_replace(
        trim(col("number")),
        r"\.0$",
        ""
    ).alias("number"),

    trim(col("unit"))
        .cast("string")
        .alias("unit"),

    trim(col("region"))
        .cast("string")
        .alias("region"),

    trim(col("district"))
        .cast("string")
        .alias("district"),

    # Geographic coordinates
    trim(col("lon"))
        .cast("double")
        .alias("lon"),

    trim(col("lat"))
        .cast("double")
        .alias("lat"),

    # Shipping address
    trim(col("ship_to_address"))
        .cast("string")
        .alias("ship_to_address"),

    # Unix timestamp -> timestamp
    from_unixtime(
        col("valid_from").cast("long")
    )
    .cast("timestamp")
    .alias("valid_from"),

    from_unixtime(
        col("valid_to").cast("long")
    )
    .cast("timestamp")
    .alias("valid_to"),

    # Purchased units
    trim(col("units_purchased"))
        .cast("int")
        .alias("units_purchased"),

    # Loyalty segment
    trim(col("loyalty_segment"))
        .cast("string")
        .alias("loyalty_segment")
)



print(f"Datensätze vor Bereinigung: { bronze_df.count()} ")
print(f"Datensätze nach Bereinigung (ohne Duplikate): {customers_silver.count()}")

display(customers_silver.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 44, Finished, Available, Finished, False)

Datensätze vor Bereinigung: 28813 
Datensätze nach Bereinigung (ohne Duplikate): 28813


SynapseWidget(Synapse.DataFrame, 52f5653a-4d04-414f-8ddc-2249d883e0e2)

# Dokumentation: Transformation & Datenbereinigung (customers)

**Layer:** Silver Layer (Cleansed / Enriched)  
**Ziel:** Standardisierung, Bereinigung und Strukturierung der Kunden-Stammdaten (*Customers*). Das Skript überführt unstrukturierte Adresskomponenten, geografische Koordinaten, Transaktionszähler und Loyalty-Segmente in ein konsistentes, relationales Format. Es ermöglicht fehlerfreie CRM-Analysen und kundenbezogene Joins im Gold Layer.

---

### 1. Eindeutigkeit & Historisierung (Deduplizierung)
* **Logik:** Über die Operation `dropDuplicates(["customer_id", "valid_from"])` wird sichergestellt, dass jede Kombination aus Kunden-ID und Gültigkeitsbeginn eindeutig bleibt. Dies bereitet die Tabelle optimal auf eine Historisierung nach dem SCD-Prinzip (Slowly Changing Dimensions) vor.

### 2. Technische Datentyp-Konvertierungen & Bereinigungen
* **Identifikatoren & Stammdaten** → `string`: Bereinigung führender oder nachfolgender Leerzeichen mittels `trim()` für alle Text-, Adress- und Steuerdaten (`customer_id`, `tax_id`, `tax_code`, `customer_name`, etc.).
* **Postleitzahlen & Hausnummern** → `string`: Bereinigung von Formatierungsartefakten (z. B. Konvertierung von Fließkommazahlen wie `46506.0` zu `46506`) mittels `regexp_replace`, um die ursprüngliche Zeichenstruktur für den Postversand beizubehalten.
* **Geografische Koordinaten** → `double`: Konvertierung von Längen- und Breitengrad (`lon` / `lat`) in hochpräzise Fließkommazahlen für Liefergebiets- und Heatmap-Analysen.
* **Gültigkeitszeiträume** → `timestamp`: Umrechnung roher Unix-Zeitstempel (`long`) mittels `from_unixtime()`, um zeitraumbezogene Zuordnungen zu Verkaufsbelegen abzusichern.
* **Kaufstatistiken** → `int`: Typkonvertierung für die kumulierte Anzahl gekaufter Einheiten (`units_purchased`).

### 3. Datenoptimierung & Persistierung
* **Persistierung:** Das Ergebnis wird im Modus `overwrite` direkt in die Zielumgebung `Retailorg_silver.customers` geschrieben.


In [44]:
from pyspark.sql.functions import col, coalesce, lit, trim, when, current_timestamp
from pyspark.sql import functions as F



# 1. Null-Wert-Behandlung basierend auf Ihrem customers_silver DataFrame
silver_customers_final = (customers_silver
    # PFLICHTSPALTE FILTERN: Zeilen ohne customer_id rigoros herausfiltern
    .filter(
        col("customer_id").isNotNull() & 
        (trim(col("customer_id")) != "") & 
        (trim(col("customer_id")) != "null")
    )
    
    # NULL-WERTE DURCH BUSINESS-STANDARDS ERSETZEN
    .select(
        col("customer_id"),
        
        # Texte standardisieren mit der clean_string_nulls Funktion
        clean_string_nulls("customer_name", "Unbekannter Kunde").alias("customer_name"),
        clean_string_nulls("loyalty_segment", "Kein Status").alias("loyalty_segment"),
        clean_string_nulls("tax_id", "Nicht hinterlegt").alias("tax_id"),
        clean_string_nulls("tax_code", "Nicht hinterlegt").alias("tax_code"),
        clean_string_nulls("state", "Unbekannt").alias("state"),
        clean_string_nulls("city", "Unbekannte Stadt").alias("city"),
        clean_string_nulls("postcode", "00000").alias("postcode"),
        clean_string_nulls("street", "Unbekannte Straße").alias("street"),
        clean_string_nulls("number", "0").alias("house_number"),
        clean_string_nulls("unit", "Standard").alias("unit"),
        clean_string_nulls("region", "Keine Region").alias("region"),
        clean_string_nulls("district", "Kein Bezirk").alias("district"),
        clean_string_nulls("ship_to_address", "Siehe Hauptadresse").alias("ship_to_address"),
        
        # Zahlen & Koordinaten absichern (Ersetzen durch 0 / 0.00, falls NULL)
        coalesce(col("lon"), lit(0.00)).alias("longitude"),
        coalesce(col("lat"), lit(0.00)).alias("latitude"),
        coalesce(col("units_purchased"), lit(0)).alias("units_purchased"),
        
        # Zeitstempel absichern (Start auf Systemzeitstempel, Ende auf unendlich/Standard-Max)
        coalesce(col("valid_from"), current_timestamp()).alias("valid_from"),
        coalesce(col("valid_to"), F.to_timestamp(lit("9999-12-31 23:59:59"))).alias("valid_to")
    )
)

# 2. In die finale Silver Delta-Tabelle schreiben
silver_customers_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("Retailorg_silver.customers")

# Protokollierung der Datensätze
print(f"Datensätze nach erster Bereinigung: {customers_silver.count()}")
print(f"Datensätze nach finaler Qualitäts-Sicherung: {silver_customers_final.count()}")

# Schema- und Ergebnisvorschau
silver_customers_final.printSchema()
display(silver_customers_final.limit(5))


StatementMeta(, 0a36bc84-252b-47c7-b71c-e5f08cb6e9d4, 46, Finished, Available, Finished, False)

Datensätze nach erster Bereinigung: 28813
Datensätze nach finaler Qualitäts-Sicherung: 28813
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- loyalty_segment: string (nullable = true)
 |-- tax_id: string (nullable = true)
 |-- tax_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- street: string (nullable = true)
 |-- house_number: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- region: string (nullable = true)
 |-- district: string (nullable = true)
 |-- ship_to_address: string (nullable = true)
 |-- longitude: double (nullable = false)
 |-- latitude: double (nullable = false)
 |-- units_purchased: integer (nullable = false)
 |-- valid_from: timestamp (nullable = false)
 |-- valid_to: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, d6048406-50e4-48b5-bfed-f12c14597855)